# Phase 1 Input Freeze And Decoder Readiness

Notebook này khóa input và kiểm tra readiness trước khi chạy `phase1_real` decoder. Notebook không train model, không chạy inference thật, và không tạo claim hiệu năng.

Mục tiêu kỹ thuật:
- Xác minh chuỗi source-of-truth từ Gate 0, Gate 1, Gate 2, Gate 2.5, Phase 0.5 preflight và Phase 0.5 estimator.
- Hash-link các artifact đầu vào sẽ được Phase 1 sử dụng.
- Kiểm tra Phase 0.5 estimator đã hoàn tất đủ điều kiện kỹ thuật: full cohort, controls complete, 200 permutations, không còn exclusion.
- Khóa denominator Phase 1: subject-level nested LOSO, 15 outer folds.
- Kiểm tra comparator readiness trước khi cho phép Phase 1 substantive run.

Giới hạn khoa học:
- Notebook này không chứng minh decoder efficacy.
- Notebook này không chứng minh privileged-transfer efficacy.
- Nếu thiếu comparator bắt buộc hoặc hash mismatch, Phase 1 substantive run phải bị chặn hoặc chỉ được chạy smoke/debug không claim.


Cập nhật governance 2026-04-20:
- Các field tên `full_phase1_substantive_run_allowed` trong notebook này là readiness kỹ thuật sơ bộ để đi tới smoke/full-run engineering scaffold, không phải authorization cho claim-bearing Phase 1.
- Sau khi A2/A2b, A2c và A2d smoke đã review xong, notebook 11 (`phase1_gap_review`) là current blocker authority trước mọi claim-bearing run.
- Nếu notebook 11 còn blocker A3/A4/final controls/calibration/influence/reporting, mọi Phase 1 run tiếp theo vẫn phải được xem là implementation/debug/non-claim.


## Cơ sở tài liệu được dùng

Các cell dưới đây bám theo các ràng buộc đã nêu trong tài liệu dự án:

- `docs/blueprint_trien_khai_v1_colab.docx`: real phase phải chạy qua CLI guard; `phase1_real` là mainline decoding + comparator suite; không để outer-test subject tham gia fit.
- `docs/V5_5_Execution_Supplement_Implementation_Annex_vi.docx`: Phase 1 chỉ chạy sau Gate 2.5; tạo outer split theo subject; outer-test cô lập; build teacher/support/reliability/admissibility từ outer-train.
- `docs/V5_5_Integrated_Proposal_vi_complete.docx`: endpoint chính là nested LOSO binary load 4 vs 8 trong maintenance; iEEG chỉ là train-time privileged information; test-time chỉ dùng scalp EEG.
- `docs/V5_5_Technical_Implementation_Spec_vi_complete.docx`: A2b, A2c, A2d là comparator bắt buộc cho headline claim; mọi learned transform phải fit trong train split; influence ceiling và calibration constraints phải được giữ.
- `docs/V5_5_Master_Artifact_Dossier_Freeze_Prereg_Reporting_Control.docx`: Phase 1 cần main report draft, calibration package, fold logs, negative controls và comparator completeness table.

Notebook chỉ operationalize các kiểm tra này. Nếu tài liệu và artifact mâu thuẫn, notebook phải báo blocker thay vì tự diễn giải.


Ghi chú vận hành mới: notebook này được giữ để khóa input/readiness nguồn; quyết định hiện tại về việc còn được mở claim-bearing run hay không phải đối chiếu thêm với `notebooks/11_colab_phase1_comparator_suite_gap_review.ipynb` và các artifact `phase1_gap_review`. Notebook 06 không được dùng một mình để vượt qua các blocker A3/A4/control/calibration/influence/reporting.


In [ ]:
# ============================================================
# Cell 1: Mount Drive and define fixed source-of-truth paths
# ============================================================
# Mục tiêu:
# - Mount Google Drive.
# - Khai báo các path đã được khóa từ các gate trước.
# - Không dùng latest.txt ở bước này vì Phase 1 phải bám đúng source-of-truth đã xác nhận.
# ============================================================

from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

REPO_DIR = Path('/content/eeg-ds004752')
REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
DATASET_ROOT = DRIVE_ROOT / 'data' / 'ds004752'

GATE0_RUN = DRIVE_ROOT / 'artifacts/gate0/20260417T102811097110Z'
GATE1_RUN = DRIVE_ROOT / 'artifacts/gate1/20260418T153918409528Z'
GATE2_RUN = DRIVE_ROOT / 'artifacts/gate2/20260418T160143330194Z'
GATE25_RUN = DRIVE_ROOT / 'artifacts/prereg/20260418T161442014597Z'
PHASE05_RUN = DRIVE_ROOT / 'artifacts/phase05/20260418T163438037205Z'
PHASE05_ESTIMATOR_RUN = DRIVE_ROOT / 'artifacts/phase05_estimators/20260419T130315366518Z'

PREREG_BUNDLE = GATE25_RUN / 'prereg_bundle.json'
EXPECTED_PREREG_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'

PHASE1_READINESS_ROOT = DRIVE_ROOT / 'artifacts/phase1_readiness'
PHASE1_READINESS_ROOT.mkdir(parents=True, exist_ok=True)

print('Repo:', REPO_DIR)
print('Dataset root:', DATASET_ROOT)
print('Gate 0 source:', GATE0_RUN)
print('Gate 1 source:', GATE1_RUN)
print('Gate 2 source:', GATE2_RUN)
print('Gate 2.5 source:', GATE25_RUN)
print('Phase 0.5 source:', PHASE05_RUN)
print('Phase 0.5 estimator source:', PHASE05_ESTIMATOR_RUN)
print('Phase 1 readiness output root:', PHASE1_READINESS_ROOT)


In [ ]:
# ============================================================
# Cell 2: Clone or update private repo, then inspect git state
# ============================================================
# Mục tiêu:
# - Đảm bảo repo tồn tại ở /content/eeg-ds004752.
# - Pull code mới nhất.
# - Không in GitHub token ra output.
# ============================================================

import base64
import getpass
import os
import subprocess
import sys


def run(cmd, cwd=None, check=True, capture=False, env=None):
    print('\n$', ' '.join(str(x) for x in cmd))
    result = subprocess.run(
        cmd,
        cwd=cwd,
        check=check,
        text=True,
        capture_output=capture,
        env=env,
    )
    if capture:
        if result.stdout:
            print(result.stdout.strip())
        if result.stderr:
            print(result.stderr.strip())
    return result


def git_auth_header():
    token = os.environ.get('GITHUB_TOKEN')
    if not token:
        token = getpass.getpass('Nhập GitHub token cho repo private: ')
    basic = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    return f'Authorization: Basic {basic}'


if not REPO_DIR.exists():
    extra_header = git_auth_header()
    run(['git', '-c', f'http.extraHeader={extra_header}', 'clone', REPO_URL, str(REPO_DIR)])
else:
    print('Repo đã tồn tại:', REPO_DIR)

os.chdir(REPO_DIR)
print('Current directory:', Path.cwd())

pull = subprocess.run(['git', 'pull'], cwd=REPO_DIR, text=True)
if pull.returncode != 0:
    extra_header = git_auth_header()
    run(['git', '-c', f'http.extraHeader={extra_header}', 'pull'], cwd=REPO_DIR)

run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)
run([sys.executable, '--version'], cwd=REPO_DIR)
run(['git', 'status', '--short'], cwd=REPO_DIR)


In [ ]:
# ============================================================
# Cell 3: Install runtime and run unit tests
# ============================================================
# Mục tiêu:
# - Cài runtime tối thiểu.
# - Cài scientific extras để kiểm tra imports nếu notebook sau cần signal/model utilities.
# - Chạy unit tests trước khi khóa Phase 1 readiness.
# ============================================================

runtime_env = os.environ.copy()
runtime_env['INSTALL_SIGNAL_EXTRAS'] = '1'

run(['bash', 'bootstrap/install_runtime.sh'], cwd=REPO_DIR, env=runtime_env)
run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests'], cwd=REPO_DIR)


In [ ]:
# ============================================================
# Cell 4: Utility functions for hash, JSON, git metadata
# ============================================================

import hashlib
import json
from datetime import datetime, timezone


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def read_json(path: Path):
    if not path.exists():
        raise FileNotFoundError(f'Missing JSON file: {path}')
    return json.loads(path.read_text(encoding='utf-8'))


def write_json(path: Path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')


def require_path(path: Path, label: str):
    if not path.exists():
        raise FileNotFoundError(f'Missing required {label}: {path}')
    return path


def git_output(args):
    return subprocess.check_output(args, cwd=REPO_DIR, text=True).strip()


def utc_stamp():
    return datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')


print('Utilities ready.')


In [ ]:
# ============================================================
# Cell 5: Verify required source-of-truth files exist
# ============================================================
# Mục tiêu:
# - Kiểm tra artifact tối thiểu của từng gate/phase.
# - Fail sớm nếu thiếu file thay vì tự suy diễn.
# ============================================================

required_files = {
    'gate0_manifest': GATE0_RUN / 'manifest.json',
    'gate0_cohort_lock': GATE0_RUN / 'cohort_lock.json',
    'gate0_materialization_report': GATE0_RUN / 'materialization_report.json',
    'gate1_summary': GATE1_RUN / 'gate1_summary.json',
    'gate1_inputs': GATE1_RUN / 'gate1_inputs.json',
    'gate1_n_eff': GATE1_RUN / 'n_eff_statement.json',
    'gate1_sesoi': GATE1_RUN / 'sesoi_registry.json',
    'gate1_influence_rule': GATE1_RUN / 'influence_rule.json',
    'gate2_summary': GATE2_RUN / 'gate2_summary.json',
    'gate2_threshold_registry': GATE2_RUN / 'gate_threshold_registry.json',
    'gate25_summary': GATE25_RUN / 'gate25_summary.json',
    'prereg_bundle': PREREG_BUNDLE,
    'environment_lock': GATE25_RUN / 'environment_lock.json',
    'real_phase_source_of_truth': GATE25_RUN / 'real_phase_source_of_truth.json',
    'real_phase_guard_smoke': GATE25_RUN / 'real_phase_guard_smoke.json',
    'phase05_summary': PHASE05_RUN / 'phase05_summary.json',
    'phase05_inputs': PHASE05_RUN / 'phase05_inputs.json',
    'phase05_estimator_summary': PHASE05_ESTIMATOR_RUN / 'phase05_estimators_summary.json',
    'phase05_feature_report': PHASE05_ESTIMATOR_RUN / 'feature_extraction_report.json',
    'phase05_controls_report': PHASE05_ESTIMATOR_RUN / 'controls_report.json',
    'phase05_teacher_survival': PHASE05_ESTIMATOR_RUN / 'teacher_survival_table.json',
    'phase05_exclusion_note': PHASE05_ESTIMATOR_RUN / 'phase05_estimator_exclusion_note.json',
    'phase05_estimator_source_of_truth': PHASE05_ESTIMATOR_RUN / 'phase05_estimators_source_of_truth.json',
}

for label, path in required_files.items():
    require_path(path, label)

print('All required source-of-truth files exist.')
for label, path in required_files.items():
    print(f'- {label}: {path}')


In [ ]:
# ============================================================
# Cell 6: Load gate summaries and verify source chain
# ============================================================

gate0_manifest = read_json(required_files['gate0_manifest'])
gate0_cohort_lock = read_json(required_files['gate0_cohort_lock'])
gate0_materialization = read_json(required_files['gate0_materialization_report'])
gate1_summary = read_json(required_files['gate1_summary'])
gate1_inputs = read_json(required_files['gate1_inputs'])
gate1_n_eff = read_json(required_files['gate1_n_eff'])
gate1_sesoi = read_json(required_files['gate1_sesoi'])
gate1_influence = read_json(required_files['gate1_influence_rule'])
gate2_summary = read_json(required_files['gate2_summary'])
gate2_thresholds = read_json(required_files['gate2_threshold_registry'])
gate25_summary = read_json(required_files['gate25_summary'])
prereg_bundle = read_json(required_files['prereg_bundle'])
phase05_summary = read_json(required_files['phase05_summary'])
phase05_inputs = read_json(required_files['phase05_inputs'])

assert gate0_manifest.get('manifest_status') == 'signal_audit_ready', gate0_manifest.get('manifest_status')
assert gate0_manifest.get('signal_audit', {}).get('status') == 'ok'
assert gate0_manifest.get('gate0_blockers') == []
assert gate1_summary.get('status') == 'gate1_decision_layer_ready'
assert gate2_summary.get('status') == 'gate2_synthetic_ready'
assert gate25_summary.get('status') == 'gate2_5_prereg_bundle_locked'
assert phase05_summary.get('status') == 'phase05_observability_preflight_ready'

source_runs = prereg_bundle.get('source_runs', {})
assert Path(source_runs.get('gate0', '')) == GATE0_RUN
assert Path(source_runs.get('gate1', '')) == GATE1_RUN
assert Path(source_runs.get('gate2', '')) == GATE2_RUN
assert prereg_bundle.get('prereg_bundle_hash_sha256') == EXPECTED_PREREG_HASH
assert gate25_summary.get('prereg_bundle_hash_sha256') == EXPECTED_PREREG_HASH
assert phase05_summary.get('prereg_bundle_hash_sha256') == EXPECTED_PREREG_HASH

print('================ Source Chain Summary ================')
print('Gate 0 status:', gate0_manifest.get('manifest_status'))
print('Gate 1 status:', gate1_summary.get('status'))
print('Gate 2 status:', gate2_summary.get('status'))
print('Gate 2.5 status:', gate25_summary.get('status'))
print('Phase 0.5 preflight status:', phase05_summary.get('status'))
print('Prereg bundle hash:', prereg_bundle.get('prereg_bundle_hash_sha256'))
print('Source chain verified: Gate 0 -> Gate 1 -> Gate 2 -> Gate 2.5 -> Phase 0.5')


In [ ]:
# ============================================================
# Cell 7: Verify prereg bundle, environment lock, guard smoke
# ============================================================
# Mục tiêu:
# - Kiểm tra prereg bundle locked và đúng expected hash.
# - Kiểm tra real_phase_source_of_truth.json có hash-link tới prereg bundle.
# - Kiểm tra real_phase_guard_smoke.json pass cho Phase 1/2/3 guard.
#
# Lưu ý schema:
# - real_phase_source_of_truth.json có thể dùng key khác nhau giữa các phiên bản notebook.
# - real_phase_guard_smoke.json ở Gate 2.5 hiện dùng key "results", không phải "guard_results".
# - phase05_real không còn guard-only vì đã chạy workflow Phase 0.5 riêng.
# ============================================================

environment_lock = read_json(required_files['environment_lock'])
real_phase_sot = read_json(required_files['real_phase_source_of_truth'])
real_phase_guard_smoke = read_json(required_files['real_phase_guard_smoke'])

# Hash file thực tế được ghi lại để truy vết. Không so sánh file SHA với embedded prereg hash.
artifact_hashes = {
    label: {
        'path': str(path),
        'sha256_file': sha256_file(path),
    }
    for label, path in required_files.items()
}

# Hard checks on the locked prereg bundle itself.
assert prereg_bundle.get('status') == 'locked'
assert prereg_bundle.get('prereg_bundle_hash_sha256') == EXPECTED_PREREG_HASH
assert gate25_summary.get('prereg_bundle_hash_sha256') == EXPECTED_PREREG_HASH

# real_phase_source_of_truth schema can vary; require that the exact locked hash is present somewhere in the record.
real_phase_sot_text = json.dumps(real_phase_sot, ensure_ascii=False, sort_keys=True)
if EXPECTED_PREREG_HASH not in real_phase_sot_text:
    print('real_phase_source_of_truth keys:', sorted(real_phase_sot.keys()))
    print(json.dumps(real_phase_sot, indent=2, ensure_ascii=False)[:4000])
    raise AssertionError('real_phase_source_of_truth.json does not contain the expected prereg bundle hash')

# Guard smoke schema handling: support both list-based "results" and dict-based "guard_results".
guard_hash = (
    real_phase_guard_smoke.get('prereg_bundle_internal_hash_sha256')
    or real_phase_guard_smoke.get('prereg_bundle_hash_sha256')
)
assert guard_hash == EXPECTED_PREREG_HASH, guard_hash

raw_guard_results = real_phase_guard_smoke.get('results')
if raw_guard_results is None:
    raw_guard_results = real_phase_guard_smoke.get('guard_results')

if isinstance(raw_guard_results, dict):
    guard_by_phase = {
        phase: item if isinstance(item, dict) else {'passed': bool(item)}
        for phase, item in raw_guard_results.items()
    }
elif isinstance(raw_guard_results, list):
    guard_by_phase = {
        item.get('phase'): item
        for item in raw_guard_results
        if isinstance(item, dict) and item.get('phase')
    }
else:
    raise AssertionError('real_phase_guard_smoke.json has neither list results nor dict guard_results')

required_guard_phases = ['phase1_real', 'phase2_real', 'phase3_real']
for phase in required_guard_phases:
    item = guard_by_phase.get(phase)
    if not item:
        raise AssertionError(f'Missing guard smoke result for {phase}')
    assert item.get('passed') is True, item

print('================ Prereg / Guard Summary ================')
print('Prereg bundle path:', PREREG_BUNDLE)
print('Prereg bundle embedded hash:', prereg_bundle.get('prereg_bundle_hash_sha256'))
print('Prereg bundle file SHA256:', artifact_hashes['prereg_bundle']['sha256_file'])
print('Environment lock file SHA256:', artifact_hashes['environment_lock']['sha256_file'])
print('Real phase SOT status:', real_phase_sot.get('status'))
print('Real phase SOT keys:', sorted(real_phase_sot.keys()))
print('Guard smoke status:', real_phase_guard_smoke.get('status'))
print('Guard phases checked:', required_guard_phases)
print('Guard smoke file:', required_files['real_phase_guard_smoke'])
print('Guard smoke note: guard acceptance only; no Phase 1/2/3 model was executed.')


In [ ]:
# ============================================================
# Cell 8: Verify final Phase 0.5 estimator closeout
# ============================================================

phase05_estimator_summary = read_json(required_files['phase05_estimator_summary'])
phase05_feature_report = read_json(required_files['phase05_feature_report'])
phase05_controls = read_json(required_files['phase05_controls_report'])
phase05_teacher_survival = read_json(required_files['phase05_teacher_survival'])
phase05_exclusion_note = read_json(required_files['phase05_exclusion_note'])
phase05_estimator_sot = read_json(required_files['phase05_estimator_source_of_truth'])

assert phase05_estimator_summary.get('status') == 'phase05_estimators_controls_complete'
assert phase05_estimator_summary.get('n_subjects') == 15
assert phase05_estimator_summary.get('n_sessions') == 68
assert phase05_estimator_summary.get('n_permutations') >= 200
assert phase05_estimator_summary.get('controls_blockers') == []
assert phase05_estimator_summary.get('phase05_observability_table_ready') is True
assert phase05_estimator_summary.get('claim_ready') is False

assert phase05_controls.get('status') == 'controls_report_complete'
assert phase05_controls.get('blockers') == []
assert phase05_controls.get('ica_control_status') == 'computed'
assert phase05_teacher_survival.get('status') == 'teacher_survival_table_ready_for_phase05_observability'

assert phase05_exclusion_note.get('selected_sessions') == 68
assert phase05_exclusion_note.get('included_sessions') == 68
assert phase05_exclusion_note.get('excluded_sessions') == 0
assert phase05_exclusion_note.get('status') == 'no_phase05_estimator_exclusions'

phase05_estimator_sot_text = json.dumps(phase05_estimator_sot, ensure_ascii=False)
assert str(PHASE05_ESTIMATOR_RUN) in phase05_estimator_sot_text
assert EXPECTED_PREREG_HASH in phase05_estimator_sot_text

print('================ Phase 0.5 Estimator Readiness ================')
print('Run:', PHASE05_ESTIMATOR_RUN)
print('Status:', phase05_estimator_summary.get('status'))
print('Subjects:', phase05_estimator_summary.get('n_subjects'))
print('Sessions:', phase05_estimator_summary.get('n_sessions'))
print('Trials:', phase05_estimator_summary.get('n_trials'))
print('Teacher targets:', phase05_estimator_summary.get('n_teacher_targets'))
print('Permutations:', phase05_estimator_summary.get('n_permutations'))
print('Controls status:', phase05_controls.get('status'))
print('ICA control status:', phase05_controls.get('ica_control_status'))
print('Controls blockers:', phase05_controls.get('blockers'))
print('Exclusions:', phase05_exclusion_note.get('excluded_sessions'))
print('Read fallbacks:', len(phase05_exclusion_note.get('read_fallbacks') or []))
print('Claim ready for privileged-transfer efficacy:', phase05_estimator_summary.get('claim_ready'))


In [ ]:
# ============================================================
# Cell 9: Freeze Phase 1 teacher candidates from Phase 0.5 table
# ============================================================

teacher_rows = phase05_teacher_survival.get('rows') or []
if not teacher_rows:
    raise ValueError('teacher_survival_table.json has no rows')


def teacher_id(row):
    return str(row.get('teacher_id') or row.get('target') or row.get('name') or '')


phase1_teacher_rows = []
unknown_group_rows = []
for row in teacher_rows:
    tid = teacher_id(row)
    explicit_group = str(row.get('group') or row.get('teacher_group') or '').lower()
    tid_lower = tid.lower()
    is_group_ab = explicit_group in {'a', 'b', 'group_a', 'group_b'} or tid_lower.startswith('group_a') or tid_lower.startswith('group_b')
    if is_group_ab:
        phase1_teacher_rows.append(row)
    else:
        unknown_group_rows.append(row)

if not phase1_teacher_rows:
    raise ValueError('No Group A/B teacher candidates detected in teacher_survival_table.json')

teacher_freeze = {
    'status': 'phase1_teacher_candidate_freeze_ready',
    'source': str(required_files['phase05_teacher_survival']),
    'source_sha256_file': sha256_file(required_files['phase05_teacher_survival']),
    'source_status': phase05_teacher_survival.get('status'),
    'total_teacher_rows': len(teacher_rows),
    'phase1_group_ab_candidate_rows': len(phase1_teacher_rows),
    'unknown_or_non_ab_rows': len(unknown_group_rows),
    'candidate_teacher_ids': [teacher_id(row) for row in phase1_teacher_rows],
    'scientific_limit': 'Teacher survival supports Phase 1 candidate freezing only; it is not decoder efficacy evidence.',
}

print('================ Phase 1 Teacher Candidate Freeze ================')
print('Status:', teacher_freeze['status'])
print('Total teacher rows:', teacher_freeze['total_teacher_rows'])
print('Group A/B candidate rows:', teacher_freeze['phase1_group_ab_candidate_rows'])
print('Unknown/non-A-B rows:', teacher_freeze['unknown_or_non_ab_rows'])
print('First 10 candidate IDs:')
for tid in teacher_freeze['candidate_teacher_ids'][:10]:
    print('-', tid)


In [ ]:
# ============================================================
# Cell 10: Comparator and claim-readiness audit
# ============================================================
# Mục tiêu:
# - Đối chiếu comparator cards/configs trong prereg bundle với yêu cầu tài liệu.
# - Báo rõ nếu thiếu comparator bắt buộc cho headline claim.
# - Không tự đổi tên, không tự map EEGNet/ShallowConvNet thành A2b/A2c nếu prereg chưa ghi rõ.
# ============================================================

comparator_cards = prereg_bundle.get('comparator_cards', {})
comparator_configs = prereg_bundle.get('config_hashes', {}).get('comparator_configs', {})
if not comparator_configs:
    comparator_configs = prereg_bundle.get('comparator_configs', {})

available_comparator_ids = sorted(set(comparator_cards.keys()) | set(comparator_configs.keys()))
required_for_guarded_phase1_scaffold = {'A2d_riemannian', 'A3_distillation', 'A4_privileged'}
required_for_headline_claim = {'A2b', 'A2c', 'A2d_riemannian', 'A3_distillation', 'A4_privileged'}

missing_scaffold = sorted(required_for_guarded_phase1_scaffold - set(available_comparator_ids))
missing_headline = sorted(required_for_headline_claim - set(available_comparator_ids))

comparator_readiness = {
    'status': 'comparator_cards_complete_for_headline_claim' if not missing_headline else 'comparator_cards_incomplete_for_headline_claim',
    'available_comparator_ids': available_comparator_ids,
    'required_for_guarded_phase1_scaffold': sorted(required_for_guarded_phase1_scaffold),
    'missing_for_guarded_phase1_scaffold': missing_scaffold,
    'required_for_headline_claim': sorted(required_for_headline_claim),
    'missing_for_headline_claim': missing_headline,
    'policy': 'Do not relabel existing comparator cards as A2b/A2c unless prereg/revision explicitly defines that mapping.',
}

assert not missing_scaffold, f'Missing scaffold comparators: {missing_scaffold}'

print('================ Comparator Readiness ================')
print('Status:', comparator_readiness['status'])
print('Available comparator IDs:', available_comparator_ids)
print('Missing for guarded Phase 1 scaffold:', missing_scaffold)
print('Missing for headline claim:', missing_headline)
if missing_headline:
    print('WARNING: Phase 1 headline privileged-value claim must remain blocked until these comparator cards are explicitly frozen or the prereg bundle is refrozen with a justified mapping.')
else:
    print('OK: Explicit comparator cards/configs are present for headline-claim comparator suite.')


In [ ]:
# ============================================================
# Cell 11: Freeze Phase 1 endpoint, split, and leakage rules
# ============================================================

assert gate1_n_eff.get('n_primary_eligible') == 15
assert gate1_n_eff.get('primary_denominator') == 'subject'
assert gate1_n_eff.get('n_outer_folds_planned') == 15
assert gate1_n_eff.get('sessions_are_primary_independent_units') is False

split_freeze = {
    'status': 'phase1_subject_level_loso_split_freeze_ready',
    'primary_endpoint': 'nested_loso_subject_level_binary_load_4_vs_8_maintenance',
    'n_primary_eligible_subjects': gate1_n_eff.get('n_primary_eligible'),
    'outer_folds_planned': gate1_n_eff.get('n_outer_folds_planned'),
    'primary_denominator': gate1_n_eff.get('primary_denominator'),
    'sessions_are_primary_independent_units': gate1_n_eff.get('sessions_are_primary_independent_units'),
    'no_leakage_rules': [
        'Outer-test subject must not participate in preprocessing fit, ICA fit, PCA fit, spatial alignment fit, latent coupling fit, observability predictor fit, calibration fit, or threshold tuning.',
        'All teacher/support/reliability/admissibility scores used for a fold must be computed from outer-train only.',
        'Inference-time input is scalp EEG only; iEEG is train-time privileged information only.',
        'LOSO-session may be secondary robustness only and cannot replace subject-level LOSO for headline claim.',
    ],
}

print('================ Phase 1 Split Freeze ================')
print('Status:', split_freeze['status'])
print('Primary endpoint:', split_freeze['primary_endpoint'])
print('N primary eligible subjects:', split_freeze['n_primary_eligible_subjects'])
print('Outer folds planned:', split_freeze['outer_folds_planned'])
print('Primary denominator:', split_freeze['primary_denominator'])
print('Sessions are primary independent units:', split_freeze['sessions_are_primary_independent_units'])


In [ ]:
# ============================================================
# Cell 12: Freeze claim thresholds and demotion rules
# ============================================================

primary_sesoi = gate1_sesoi.get('primary_subject_level_sesoi', {})
calibration_tolerance = gate1_sesoi.get('calibration_tolerance', {})

claim_thresholds = {
    'status': 'phase1_claim_thresholds_frozen_from_gate1',
    'primary_metric': primary_sesoi.get('metric'),
    'primary_comparison': primary_sesoi.get('comparison'),
    'subject_level_sesoi_delta_ba': primary_sesoi.get('median_delta_ba_min'),
    'calibration_metric': calibration_tolerance.get('metric'),
    'max_allowed_delta_ece': calibration_tolerance.get('max_allowed_delta_ece'),
    'influence_ceiling': gate1_influence.get('influence_ceiling'),
    'strong_claim_requires': gate1_sesoi.get('strong_claim_requires', []),
    'hard_limits': [
        'A4 must exceed A3 and required scalp-only comparators including A2d for strong privileged-value claim.',
        'A2b/A2c/A2d completeness is mandatory before headline claim.',
        'Calibration deterioration must not exceed locked threshold.',
        'Influence concentration above ceiling blocks State A even if nominal p-value passes.',
        'Negative controls must pass; shuffled/time-shifted teacher gains must collapse.',
    ],
}

print('================ Claim Threshold Freeze ================')
print('Status:', claim_thresholds['status'])
print('Primary metric:', claim_thresholds['primary_metric'])
print('Primary comparison:', claim_thresholds['primary_comparison'])
print('Subject-level SESOI Delta BA:', claim_thresholds['subject_level_sesoi_delta_ba'])
print('Max allowed Delta ECE:', claim_thresholds['max_allowed_delta_ece'])
print('Influence ceiling:', claim_thresholds['influence_ceiling'])
print('Strong claim requires:', claim_thresholds['strong_claim_requires'])


In [ ]:
# ============================================================
# Cell 13: Phase 1 CLI guard smoke
# ============================================================

phase1_guard_cmd = [
    sys.executable,
    '-m',
    'src.cli',
    'phase1_real',
    '--profile',
    'a100_fast',
    '--config',
    str(PREREG_BUNDLE),
]

result = subprocess.run(phase1_guard_cmd, cwd=REPO_DIR, text=True, capture_output=True)
print('$', ' '.join(phase1_guard_cmd))
print(result.stdout.strip())
if result.stderr.strip():
    print(result.stderr.strip())

phase1_guard_smoke = {
    'status': 'phase1_real_guard_passed' if result.returncode == 0 else 'phase1_real_guard_failed',
    'command': phase1_guard_cmd,
    'returncode': result.returncode,
    'stdout': result.stdout,
    'stderr': result.stderr,
    'prereg_bundle': str(PREREG_BUNDLE),
    'prereg_bundle_hash_sha256': EXPECTED_PREREG_HASH,
    'does_not_train_decoder': True,
    'does_not_compute_real_model_results': True,
}

assert result.returncode == 0, phase1_guard_smoke
print('Guard status:', phase1_guard_smoke['status'])
print('Note: This confirms guard acceptance only; no Phase 1 decoder was executed.')


In [ ]:
# ============================================================
# Cell 14: Write Phase 1 input freeze/readiness record
# ============================================================

run_dir = PHASE1_READINESS_ROOT / utc_stamp()
run_dir.mkdir(parents=True, exist_ok=False)

repo_branch = git_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD'])
repo_commit = git_output(['git', 'rev-parse', 'HEAD'])
git_status_short = git_output(['git', 'status', '--short'])

headliner_comparator_complete = comparator_readiness['missing_for_headline_claim'] == []
phase05_ready = (
    phase05_estimator_summary.get('status') == 'phase05_estimators_controls_complete'
    and phase05_estimator_summary.get('phase05_observability_table_ready') is True
    and phase05_controls.get('blockers') == []
    and phase05_exclusion_note.get('included_sessions') == 68
    and phase05_exclusion_note.get('excluded_sessions') == 0
)

readiness_status = (
    'phase1_input_freeze_ready_for_decoder_smoke_and_full_comparator_run'
    if headliner_comparator_complete and phase05_ready
    else 'phase1_input_freeze_recorded_with_blockers_or_warnings'
)

phase1_input_freeze = {
    'status': readiness_status,
    'created_utc': run_dir.name,
    'repo': {
        'path': str(REPO_DIR),
        'branch': repo_branch,
        'commit': repo_commit,
        'working_tree_clean': git_status_short == '',
        'git_status_short': git_status_short,
    },
    'source_of_truth': {
        'gate0': str(GATE0_RUN),
        'gate1': str(GATE1_RUN),
        'gate2': str(GATE2_RUN),
        'gate25': str(GATE25_RUN),
        'phase05_preflight': str(PHASE05_RUN),
        'phase05_estimators': str(PHASE05_ESTIMATOR_RUN),
        'prereg_bundle': str(PREREG_BUNDLE),
        'prereg_bundle_hash_sha256': EXPECTED_PREREG_HASH,
    },
    'artifact_hashes_sha256_file': artifact_hashes,
    'phase05_estimator_readiness': {
        'status': phase05_estimator_summary.get('status'),
        'n_subjects': phase05_estimator_summary.get('n_subjects'),
        'n_sessions': phase05_estimator_summary.get('n_sessions'),
        'n_trials': phase05_estimator_summary.get('n_trials'),
        'n_teacher_targets': phase05_estimator_summary.get('n_teacher_targets'),
        'n_permutations': phase05_estimator_summary.get('n_permutations'),
        'controls_status': phase05_controls.get('status'),
        'controls_blockers': phase05_controls.get('blockers'),
        'selected_sessions': phase05_exclusion_note.get('selected_sessions'),
        'included_sessions': phase05_exclusion_note.get('included_sessions'),
        'excluded_sessions': phase05_exclusion_note.get('excluded_sessions'),
        'read_fallbacks': phase05_exclusion_note.get('read_fallbacks') or [],
        'claim_ready_for_privileged_transfer_efficacy': phase05_estimator_summary.get('claim_ready'),
    },
    'teacher_candidate_freeze': teacher_freeze,
    'comparator_readiness': comparator_readiness,
    'split_freeze': split_freeze,
    'claim_thresholds': claim_thresholds,
    'phase1_guard_smoke': phase1_guard_smoke,
    'authorization': {
        'decoder_smoke_allowed_under_guard': phase1_guard_smoke['status'] == 'phase1_real_guard_passed' and phase05_ready,
        'full_phase1_substantive_run_allowed': phase1_guard_smoke['status'] == 'phase1_real_guard_passed' and phase05_ready and headliner_comparator_complete,
        'full_phase1_substantive_run_allowed_scope': 'pre_gap_review_engineering_readiness_only_not_claim_bearing_authorization',
        'claim_bearing_phase1_run_allowed_from_this_record': False,
        'current_gap_review_required_before_claim_bearing_phase1': True,
        'current_authority_for_claim_bearing_phase1': 'notebooks/11_colab_phase1_comparator_suite_gap_review.ipynb and phase1_gap_review artifacts',
        'headline_privileged_value_claim_allowed_from_this_notebook': False,
        'if_comparator_cards_missing': 'Do not run/claim full Phase 1 headline analysis until explicit A2b/A2c comparator cards or prereg-approved mapping are locked.',
    },
    'scientific_integrity_limits': [
        'This readiness record does not train a decoder.',
        'This readiness record does not compute Phase 1 performance.',
        'This readiness record does not prove privileged-transfer efficacy.',
        'The full_phase1_substantive_run_allowed flag in this notebook is pre-gap-review engineering readiness only.',
        'Notebook 11 / phase1_gap_review is the current authority for remaining A3/A4/control/calibration/influence/reporting blockers.',
        'Any claim-affecting change after prereg must be logged and treated as post-hoc unless refrozen and rerun.',
    ],
}

freeze_path = run_dir / 'phase1_input_freeze.json'
write_json(freeze_path, phase1_input_freeze)

latest_path = PHASE1_READINESS_ROOT / 'latest.txt'
latest_path.write_text(str(run_dir), encoding='utf-8')

print('================ Phase 1 Input Freeze Written ================')
print('Run dir:', run_dir)
print('Freeze file:', freeze_path)
print('Status:', phase1_input_freeze['status'])
print('Decoder smoke allowed:', phase1_input_freeze['authorization']['decoder_smoke_allowed_under_guard'])
print('Full Phase 1 substantive run allowed (pre-gap-review engineering only):', phase1_input_freeze['authorization']['full_phase1_substantive_run_allowed'])
print('Claim-bearing Phase 1 run allowed from this record:', phase1_input_freeze['authorization']['claim_bearing_phase1_run_allowed_from_this_record'])
print('Current gap review required before claim-bearing Phase 1:', phase1_input_freeze['authorization']['current_gap_review_required_before_claim_bearing_phase1'])
print('Headline claim allowed from this notebook:', phase1_input_freeze['authorization']['headline_privileged_value_claim_allowed_from_this_notebook'])
print('Working tree clean:', phase1_input_freeze['repo']['working_tree_clean'])


In [ ]:
# ============================================================
# Cell 15: Render readiness report
# ============================================================

report_lines = [
    '# Phase 1 Input Freeze And Decoder Readiness Report',
    '',
    '## Status',
    '',
    f'- Status: `{phase1_input_freeze["status"]}`',
    f'- Run dir: `{run_dir}`',
    f'- Prereg bundle hash: `{EXPECTED_PREREG_HASH}`',
    f'- Git commit: `{repo_commit}`',
    f'- Working tree clean: `{phase1_input_freeze["repo"]["working_tree_clean"]}`',
    '',
    '## Source Of Truth',
    '',
]
for key, value in phase1_input_freeze['source_of_truth'].items():
    report_lines.append(f'- {key}: `{value}`')

report_lines.extend([
    '',
    '## Phase 0.5 Estimator Readiness',
    '',
    f'- Status: `{phase05_estimator_summary.get("status")}`',
    f'- Subjects: {phase05_estimator_summary.get("n_subjects")}',
    f'- Sessions: {phase05_estimator_summary.get("n_sessions")}',
    f'- Trials: {phase05_estimator_summary.get("n_trials")}',
    f'- Teacher targets: {phase05_estimator_summary.get("n_teacher_targets")}',
    f'- Permutations: {phase05_estimator_summary.get("n_permutations")}',
    f'- Controls blockers: `{phase05_controls.get("blockers")}`',
    f'- Included sessions: {phase05_exclusion_note.get("included_sessions")}',
    f'- Excluded sessions: {phase05_exclusion_note.get("excluded_sessions")}',
    '',
    '## Comparator Readiness',
    '',
    f'- Status: `{comparator_readiness["status"]}`',
    f'- Available comparator IDs: `{comparator_readiness["available_comparator_ids"]}`',
    f'- Missing for headline claim: `{comparator_readiness["missing_for_headline_claim"]}`',
    '',
    '## Authorization',
    '',
    f'- Decoder smoke allowed under guard: `{phase1_input_freeze["authorization"]["decoder_smoke_allowed_under_guard"]}`',
    f'- Full Phase 1 substantive run allowed: `{phase1_input_freeze["authorization"]["full_phase1_substantive_run_allowed"]}`',
    f'- Full Phase 1 run scope: `{phase1_input_freeze["authorization"]["full_phase1_substantive_run_allowed_scope"]}`',
    f'- Claim-bearing Phase 1 run allowed from this record: `{phase1_input_freeze["authorization"]["claim_bearing_phase1_run_allowed_from_this_record"]}`',
    f'- Current gap review required before claim-bearing Phase 1: `{phase1_input_freeze["authorization"]["current_gap_review_required_before_claim_bearing_phase1"]}`',
    f'- Current authority for claim-bearing Phase 1: `{phase1_input_freeze["authorization"]["current_authority_for_claim_bearing_phase1"]}`',
    f'- Headline privileged-value claim allowed from this notebook: `{phase1_input_freeze["authorization"]["headline_privileged_value_claim_allowed_from_this_notebook"]}`',
    '',
    '## Scientific Integrity',
    '',
    '- This report does not train a decoder.',
    '- This report does not compute Phase 1 model performance.',
    '- This report does not prove privileged-transfer efficacy.',
    '- If comparator cards are incomplete, only smoke/debug implementation should proceed until prereg-compliant comparator completeness is restored.',
    '- Even if this readiness record says full Phase 1 substantive run allowed, that field is pre-gap-review engineering readiness only.',
    '- Notebook 11 / phase1_gap_review must be checked before any claim-bearing Phase 1 run; unresolved A3/A4/control/calibration/influence/reporting blockers keep claims closed.',
])

report_path = run_dir / 'phase1_readiness_report.md'
report_path.write_text('\n'.join(report_lines) + '\n', encoding='utf-8')

print('================ Report Preview ================')
print(report_path.read_text(encoding='utf-8')[:5000])
print('\nReport written:', report_path)


In [ ]:
# ============================================================
# Cell 16: Final interpretation and next action
# ============================================================

print('================ Final Interpretation ================')
print('Phase 1 readiness source of truth:', run_dir)
print('Status:', phase1_input_freeze['status'])
print('Decoder smoke allowed:', phase1_input_freeze['authorization']['decoder_smoke_allowed_under_guard'])
print('Full Phase 1 substantive run allowed (pre-gap-review engineering only):', phase1_input_freeze['authorization']['full_phase1_substantive_run_allowed'])
print('Claim-bearing Phase 1 run allowed from this record:', phase1_input_freeze['authorization']['claim_bearing_phase1_run_allowed_from_this_record'])
print('Current claim-bearing authority:', phase1_input_freeze['authorization']['current_authority_for_claim_bearing_phase1'])

if comparator_readiness['missing_for_headline_claim']:
    print('CHECK REQUIRED: Explicit headline comparator cards/configs are incomplete:', comparator_readiness['missing_for_headline_claim'])
    print('NEXT: Add/freeze explicit A2b and A2c comparator cards or create a prereg/revision-approved mapping before full Phase 1 headline run.')
    print('ALLOWED: Phase 1 implementation smoke/debug may proceed only if labeled non-claim and guarded by this readiness record.')
else:
    print('OK: Comparator cards/configs required for headline-readiness are explicitly present.')
    print('NEXT: Create/run Phase 1 decoder smoke notebook on 1-2 outer folds before full 15-fold run.')

print('CURRENT GOVERNANCE UPDATE: after A2/A2b, A2c and A2d smoke reviews, notebook 11 / phase1_gap_review is required before any claim-bearing Phase 1 run.')
print('NOT OK TO CLAIM: This notebook did not train a decoder and did not show privileged-transfer efficacy.')
print('All Phase 1 outputs must remain tied to this readiness file, a newer refrozen readiness file, and the current gap-review artifacts.')


## Gate 2.5 Comparator Revision / Refreeze Addendum

Các cell sau xử lý blocker `A2b/A2c` mà Phase 1 readiness đã phát hiện. Đây là addendum/refreeze ở mức comparator mapping, không thay threshold, SESOI, teacher pool, control suite hay kết quả Phase 0.5.

Cơ sở từ docs:
- `A2b`: pooled scalp-only cross-subject baseline.
- `A2c`: scalp-only domain-generalization baseline using CORAL.
- `EEGNet`: backbone mainline có thể dùng cho A2/A2b/A2c/A3/A4 khi phù hợp, nhưng không được tự coi là comparator ID nếu chưa có mapping.
- `CORAL`: objective/domain-generalization route cho A2c.

Mục tiêu của addendum là tạo mapping/card rõ ràng và hash-link config hiện có trước khi Phase 1 full run được mở.


In [ ]:
# ============================================================
# Cell 17: Create Gate 2.5 comparator revision/refreeze registry
# ============================================================
# Mục tiêu:
# - Tạo comparator revision registry cho A2b và A2c.
# - Ghi rõ mapping dựa trên docs, không tự diễn giải hậu nghiệm.
# - Hash-link config/card liên quan.
# - Không thay đổi threshold, SESOI, teacher pool, admissibility, controls.
#
# Mapping được khóa:
# - A2b = pooled scalp-only cross-subject baseline, dùng scalp EEG only, L_task only.
# - A2c = scalp-only CORAL domain-generalization baseline, dùng scalp EEG only, L_task + beta * L_CORAL.
#
# Hash policy:
# - Không ghi SHA256 file của chính registry vào trong registry vì đó là self-referential hash.
# - Thay vào đó ghi registry_content_hash_sha256 tính trên object registry sau khi loại trường hash này.
# ============================================================

COMPARATOR_REVISION_DIR = GATE25_RUN / 'phase1_comparator_revision'
COMPARATOR_REVISION_DIR.mkdir(parents=True, exist_ok=True)
COMPARATOR_REVISION_CARDS_DIR = COMPARATOR_REVISION_DIR / 'comparator_cards'
COMPARATOR_REVISION_CARDS_DIR.mkdir(parents=True, exist_ok=True)


def sha256_json_content(data: dict, excluded_keys=None) -> str:
    excluded_keys = set(excluded_keys or [])
    clean = {k: v for k, v in data.items() if k not in excluded_keys}
    payload = json.dumps(clean, sort_keys=True, ensure_ascii=False, separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()


# Configs đã tồn tại trong repo. Cell này chỉ hash-link, không sửa nội dung config.
config_paths = {
    'A2b_backbone_eegnet': REPO_DIR / 'configs/models/eegnet.yaml',
    'A2c_backbone_eegnet': REPO_DIR / 'configs/models/eegnet.yaml',
    'A2c_coral_objective': REPO_DIR / 'configs/models/coral.yaml',
    'A2d_riemannian': REPO_DIR / 'configs/models/riemannian_a2d.yaml',
    'A3_distillation': REPO_DIR / 'configs/models/distill_a3.yaml',
    'A4_privileged': REPO_DIR / 'configs/models/privileged_a4.yaml',
}

for label, path in config_paths.items():
    if not path.exists():
        raise FileNotFoundError(f'Missing comparator config for {label}: {path}')

config_hashes = {
    label: {
        'path': str(path.relative_to(REPO_DIR)),
        'sha256_file': sha256_file(path),
    }
    for label, path in config_paths.items()
}

revision_cards = {
    'A2b': {
        'comparator_id': 'A2b',
        'canonical_name': 'A2b_pooled_scalp_cross_subject',
        'definition_from_docs': 'Pooled scalp-only cross-subject baseline.',
        'input_representation': 'scalp EEG only at train and test time',
        'privileged_information': 'not used',
        'teacher_information': 'not used',
        'backbone_family': 'EEGNet unless a prereg-compatible implementation replaces the placeholder through revision/refreeze',
        'loss': 'L_task only',
        'split_discipline': 'nested subject-level LOSO; all fit/tuning/calibration on outer-train/inner splits only; no outer-test adaptation',
        'calibration_route': 'same Phase 1 calibration route as other comparators, selected only on inner validation',
        'claim_role': 'required scalp-only comparator for headline privileged-value claim',
        'config_hashes': {
            'backbone': config_hashes['A2b_backbone_eegnet'],
        },
        'scientific_limit': 'This card freezes comparator identity and config hash only; it does not implement or run the comparator.',
    },
    'A2c': {
        'comparator_id': 'A2c',
        'canonical_name': 'A2c_coral_scalp_domain_generalization',
        'definition_from_docs': 'Scalp-only domain-generalization baseline using CORAL.',
        'input_representation': 'scalp EEG only at train and test time',
        'privileged_information': 'not used',
        'teacher_information': 'not used',
        'backbone_family': 'EEGNet backbone with CORAL objective unless a prereg-compatible implementation replaces the placeholder through revision/refreeze',
        'loss': 'L_task + beta * L_CORAL',
        'domain_pairing_rule': 'subject/domain pairing and beta selection must be fit/tuned only inside outer-train inner CV',
        'split_discipline': 'nested subject-level LOSO; no outer-test subject in CORAL alignment, beta tuning, calibration, or learned transforms',
        'calibration_route': 'same Phase 1 calibration route as other comparators, selected only on inner validation',
        'claim_role': 'required scalp-only domain-generalization comparator for headline privileged-value claim',
        'config_hashes': {
            'backbone': config_hashes['A2c_backbone_eegnet'],
            'coral_objective': config_hashes['A2c_coral_objective'],
        },
        'scientific_limit': 'This card freezes comparator identity and config hash only; it does not implement or run the comparator.',
    },
}

card_paths = {}
for comparator_id, card in revision_cards.items():
    card_path = COMPARATOR_REVISION_CARDS_DIR / f'{comparator_id}.json'
    write_json(card_path, card)
    card_paths[comparator_id] = {
        'path': str(card_path),
        'sha256_file': sha256_file(card_path),
    }

revision_registry = {
    'status': 'phase1_comparator_revision_locked',
    'created_utc': utc_stamp(),
    'base_gate25_source_of_truth': str(GATE25_RUN),
    'base_prereg_bundle': str(PREREG_BUNDLE),
    'base_prereg_bundle_hash_sha256': EXPECTED_PREREG_HASH,
    'revision_type': 'comparator_mapping_refreeze_addendum',
    'reason': 'Phase 1 readiness detected missing explicit A2b/A2c comparator identities required for headline-claim comparator completeness.',
    'doc_basis': {
        'A2b': 'Pooled scalp-only cross-subject baseline.',
        'A2c': 'Scalp-only domain-generalization baseline using CORAL.',
        'EEGNet': 'Backbone mainline usable for A2/A2b/A2c/A3/A4 when appropriate; not itself a comparator ID without this mapping.',
        'CORAL': 'Objective/domain-generalization route for A2c.',
    },
    'does_not_change': [
        'Gate 0 cohort lock',
        'Gate 1 SESOI thresholds',
        'Gate 1 influence rule',
        'Gate 2 threshold registry',
        'teacher pool',
        'admissibility rubric',
        'control suite',
        'Phase 0.5 estimator outputs',
    ],
    'comparator_cards': card_paths,
    'config_hashes': config_hashes,
    'mapping': {
        'A2b': revision_cards['A2b'],
        'A2c': revision_cards['A2c'],
    },
    'allowed_use': 'May satisfy Phase 1 comparator identity/readiness only when hash-linked by phase1_input_freeze_revision.json.',
    'not_allowed_use': 'Does not prove comparator implementation correctness, decoder performance, or privileged-transfer efficacy.',
}

revision_registry['registry_content_hash_sha256'] = sha256_json_content(
    revision_registry,
    excluded_keys={'registry_content_hash_sha256'},
)

registry_path = COMPARATOR_REVISION_DIR / 'phase1_comparator_revision_registry.json'
write_json(registry_path, revision_registry)

# Optional sidecar file hash for filesystem integrity. It is outside the registry to avoid self-reference.
registry_file_hash_path = COMPARATOR_REVISION_DIR / 'phase1_comparator_revision_registry.sha256'
registry_file_hash_path.write_text(sha256_file(registry_path) + '  phase1_comparator_revision_registry.json\n', encoding='utf-8')

print('================ Comparator Revision Registry ================')
print('Status:', revision_registry['status'])
print('Registry:', registry_path)
print('Registry content hash:', revision_registry['registry_content_hash_sha256'])
print('Registry file SHA256 sidecar:', registry_file_hash_path)
print('Base prereg hash:', revision_registry['base_prereg_bundle_hash_sha256'])
print('Cards:')
for name, item in revision_registry['comparator_cards'].items():
    print('-', name, item['path'], item['sha256_file'])
print('Does not change:', revision_registry['does_not_change'])


In [ ]:
# ============================================================
# Cell 18: Recompute comparator readiness with revision registry
# ============================================================
# Mục tiêu:
# - Đọc comparator revision registry vừa khóa.
# - Merge A2b/A2c vào comparator readiness như một prereg addendum.
# - Không sửa prereg_bundle.json gốc.
# ============================================================

registry_path = COMPARATOR_REVISION_DIR / 'phase1_comparator_revision_registry.json'
revision_registry = read_json(registry_path)
assert revision_registry['status'] == 'phase1_comparator_revision_locked'
assert revision_registry['base_prereg_bundle_hash_sha256'] == EXPECTED_PREREG_HASH

# Verify content hash, excluding the self-hash field itself.
assert revision_registry['registry_content_hash_sha256'] == sha256_json_content(
    revision_registry,
    excluded_keys={'registry_content_hash_sha256'},
)

# Verify optional sidecar file hash if present.
registry_file_hash_path = COMPARATOR_REVISION_DIR / 'phase1_comparator_revision_registry.sha256'
if registry_file_hash_path.exists():
    expected_file_hash = registry_file_hash_path.read_text(encoding='utf-8').split()[0]
    assert expected_file_hash == sha256_file(registry_path), expected_file_hash

base_available = set(available_comparator_ids)
revision_available = set(revision_registry.get('mapping', {}).keys())
revised_available_comparator_ids = sorted(base_available | revision_available)

required_for_headline_claim = {'A2b', 'A2c', 'A2d_riemannian', 'A3_distillation', 'A4_privileged'}
revised_missing_headline = sorted(required_for_headline_claim - set(revised_available_comparator_ids))

revised_comparator_readiness = {
    'status': 'comparator_cards_complete_for_headline_claim' if not revised_missing_headline else 'comparator_cards_incomplete_for_headline_claim',
    'base_available_comparator_ids': sorted(base_available),
    'revision_available_comparator_ids': sorted(revision_available),
    'available_comparator_ids_after_revision': revised_available_comparator_ids,
    'required_for_headline_claim': sorted(required_for_headline_claim),
    'missing_for_headline_claim_after_revision': revised_missing_headline,
    'revision_registry': str(registry_path),
    'revision_registry_content_hash_sha256': revision_registry['registry_content_hash_sha256'],
    'revision_registry_file_sha256': sha256_file(registry_path),
    'policy': 'A2b/A2c are accepted only through this hash-linked revision registry; no unrecorded relabeling is allowed.',
}

assert revised_comparator_readiness['status'] == 'comparator_cards_complete_for_headline_claim', revised_comparator_readiness

print('================ Revised Comparator Readiness ================')
print('Status:', revised_comparator_readiness['status'])
print('Base comparator IDs:', revised_comparator_readiness['base_available_comparator_ids'])
print('Revision comparator IDs:', revised_comparator_readiness['revision_available_comparator_ids'])
print('Available after revision:', revised_comparator_readiness['available_comparator_ids_after_revision'])
print('Missing after revision:', revised_comparator_readiness['missing_for_headline_claim_after_revision'])
print('Revision registry:', registry_path)
print('Revision registry content hash:', revised_comparator_readiness['revision_registry_content_hash_sha256'])
print('Revision registry file SHA256:', revised_comparator_readiness['revision_registry_file_sha256'])


In [ ]:
# ============================================================
# Cell 19: Write revised Phase 1 readiness source-of-truth
# ============================================================
# Mục tiêu:
# - Tạo Phase 1 readiness run mới có hash-link comparator revision registry.
# - Ghi readiness kỹ thuật cho Phase 1 scaffold nếu các điều kiện khác vẫn pass; không cho phép claim-bearing run từ notebook này.
# - Vẫn không cho phép claim từ notebook này vì chưa train decoder.
# ============================================================

revised_run_dir = PHASE1_READINESS_ROOT / utc_stamp()
revised_run_dir.mkdir(parents=True, exist_ok=False)

repo_branch = git_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD'])
repo_commit = git_output(['git', 'rev-parse', 'HEAD'])
git_status_short = git_output(['git', 'status', '--short'])

phase05_ready = (
    phase05_estimator_summary.get('status') == 'phase05_estimators_controls_complete'
    and phase05_estimator_summary.get('phase05_observability_table_ready') is True
    and phase05_controls.get('blockers') == []
    and phase05_exclusion_note.get('included_sessions') == 68
    and phase05_exclusion_note.get('excluded_sessions') == 0
)

revised_full_allowed = (
    phase1_guard_smoke['status'] == 'phase1_real_guard_passed'
    and phase05_ready
    and revised_comparator_readiness['status'] == 'comparator_cards_complete_for_headline_claim'
)

phase1_input_freeze_revision = {
    'status': 'phase1_input_freeze_revised_comparator_complete' if revised_full_allowed else 'phase1_input_freeze_revision_recorded_with_blockers',
    'created_utc': revised_run_dir.name,
    'supersedes_readiness_run': str(run_dir),
    'repo': {
        'path': str(REPO_DIR),
        'branch': repo_branch,
        'commit': repo_commit,
        'working_tree_clean': git_status_short == '',
        'git_status_short': git_status_short,
    },
    'source_of_truth': {
        'gate0': str(GATE0_RUN),
        'gate1': str(GATE1_RUN),
        'gate2': str(GATE2_RUN),
        'gate25': str(GATE25_RUN),
        'base_prereg_bundle': str(PREREG_BUNDLE),
        'base_prereg_bundle_hash_sha256': EXPECTED_PREREG_HASH,
        'phase05_preflight': str(PHASE05_RUN),
        'phase05_estimators': str(PHASE05_ESTIMATOR_RUN),
        'phase1_comparator_revision_registry': str(registry_path),
        'phase1_comparator_revision_registry_content_hash_sha256': revision_registry['registry_content_hash_sha256'],
        'phase1_comparator_revision_registry_file_sha256': sha256_file(registry_path),
    },
    'phase05_estimator_readiness': phase1_input_freeze['phase05_estimator_readiness'],
    'teacher_candidate_freeze': teacher_freeze,
    'base_comparator_readiness': comparator_readiness,
    'revised_comparator_readiness': revised_comparator_readiness,
    'split_freeze': split_freeze,
    'claim_thresholds': claim_thresholds,
    'phase1_guard_smoke': phase1_guard_smoke,
    'authorization': {
        'decoder_smoke_allowed_under_guard': phase1_guard_smoke['status'] == 'phase1_real_guard_passed' and phase05_ready,
        'full_phase1_substantive_run_allowed': revised_full_allowed,
        'full_phase1_substantive_run_allowed_scope': 'pre_gap_review_engineering_readiness_only_not_claim_bearing_authorization',
        'claim_bearing_phase1_run_allowed_from_this_revision': False,
        'current_gap_review_required_before_claim_bearing_phase1': True,
        'current_authority_for_claim_bearing_phase1': 'notebooks/11_colab_phase1_comparator_suite_gap_review.ipynb and phase1_gap_review artifacts',
        'headline_privileged_value_claim_allowed_from_this_notebook': False,
        'headline_claim_requires_future_phase1_results': True,
        'required_future_checks': [
            'A2b/A2c/A2d/A3/A4 implementations must follow the locked split discipline.',
            'Comparator completeness table must pass in Phase 1 report.',
            'Calibration, influence, CI/permutation and negative controls must pass before any claim state upgrade.',
        ],
    },
    'scientific_integrity_limits': [
        'This revision locks comparator identity/mapping only.',
        'This revision does not train a decoder.',
        'This revision does not compute Phase 1 performance.',
        'This revision does not prove privileged-transfer efficacy.',
        'The full_phase1_substantive_run_allowed flag in this revision is pre-gap-review engineering readiness only.',
        'Notebook 11 / phase1_gap_review is the current authority for remaining A3/A4/control/calibration/influence/reporting blockers.',
        'Any later implementation change affecting comparator fairness requires another revision/refreeze or post-hoc demotion.',
    ],
}

revision_freeze_path = revised_run_dir / 'phase1_input_freeze_revision.json'
write_json(revision_freeze_path, phase1_input_freeze_revision)
(PHASE1_READINESS_ROOT / 'latest.txt').write_text(str(revised_run_dir), encoding='utf-8')

report_lines = [
    '# Phase 1 Input Freeze Comparator Revision Report',
    '',
    f'- Status: `{phase1_input_freeze_revision["status"]}`',
    f'- Supersedes readiness run: `{run_dir}`',
    f'- Comparator revision registry: `{registry_path}`',
    f'- Comparator revision registry content hash: `{revision_registry["registry_content_hash_sha256"]}`',
    f'- Comparator revision registry file SHA256: `{sha256_file(registry_path)}`',
    f'- Full Phase 1 substantive run allowed: `{revised_full_allowed}`',
    f'- Full Phase 1 run scope: `{phase1_input_freeze_revision["authorization"]["full_phase1_substantive_run_allowed_scope"]}`',
    f'- Claim-bearing Phase 1 run allowed from this revision: `{phase1_input_freeze_revision["authorization"]["claim_bearing_phase1_run_allowed_from_this_revision"]}`',
    f'- Current gap review required before claim-bearing Phase 1: `{phase1_input_freeze_revision["authorization"]["current_gap_review_required_before_claim_bearing_phase1"]}`',
    f'- Current authority for claim-bearing Phase 1: `{phase1_input_freeze_revision["authorization"]["current_authority_for_claim_bearing_phase1"]}`',
    f'- Headline claim allowed from this notebook: `False`',
    '',
    '## Comparator Mapping',
    '',
    '- A2b: pooled scalp-only cross-subject baseline, scalp EEG only, L_task only.',
    '- A2c: scalp-only CORAL domain-generalization baseline, scalp EEG only, L_task + beta * L_CORAL.',
    '',
    '## Scientific Limits',
    '',
    '- This addendum does not run or validate comparator implementations.',
    '- This addendum does not train a Phase 1 decoder.',
    '- This addendum does not prove privileged-transfer efficacy.',
]
revision_report_path = revised_run_dir / 'phase1_input_freeze_revision_report.md'
revision_report_path.write_text('\n'.join(report_lines) + '\n', encoding='utf-8')

print('================ Revised Phase 1 Readiness Written ================')
print('Run dir:', revised_run_dir)
print('Freeze revision:', revision_freeze_path)
print('Report:', revision_report_path)
print('Status:', phase1_input_freeze_revision['status'])
print('Decoder smoke allowed:', phase1_input_freeze_revision['authorization']['decoder_smoke_allowed_under_guard'])
print('Full Phase 1 substantive run allowed:', phase1_input_freeze_revision['authorization']['full_phase1_substantive_run_allowed'])
print('Headline claim allowed from this notebook:', phase1_input_freeze_revision['authorization']['headline_privileged_value_claim_allowed_from_this_notebook'])


In [ ]:
# ============================================================
# Cell 20: Final interpretation after comparator revision
# ============================================================

print('================ Final Interpretation After Comparator Revision ================')
print('Revised Phase 1 readiness source of truth:', revised_run_dir)
print('Status:', phase1_input_freeze_revision['status'])
print('Comparator revision registry:', registry_path)
print('Full Phase 1 substantive run allowed (pre-gap-review engineering only):', phase1_input_freeze_revision['authorization']['full_phase1_substantive_run_allowed'])
print('Claim-bearing Phase 1 run allowed from this revision:', phase1_input_freeze_revision['authorization']['claim_bearing_phase1_run_allowed_from_this_revision'])
print('Current gap review required before claim-bearing Phase 1:', phase1_input_freeze_revision['authorization']['current_gap_review_required_before_claim_bearing_phase1'])
print('NOT OK TO CLAIM: No Phase 1 decoder has been trained and no privileged-transfer efficacy has been shown.')
print('NEXT: Create Phase 1 decoder smoke notebook and run only 1-2 outer folds first, tied to this revised readiness source-of-truth.')
print('CURRENT AUTHORITY: After smoke reviews, notebook 11 / phase1_gap_review must remain the blocker authority before any claim-bearing Phase 1 run.')
print('REQUIRED LATER: Full 15-fold run must produce comparator completeness table, calibration package, fold logs, negative controls, influence report and claim-state summary before any claim is considered.')
